In [8]:
from torchsummary import summary
from python_lib.ecapa_classifier import *
from python_lib.trainers import *
import torch
import numpy as np
import os
import h5py
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torch.utils.data import random_split
from datetime import datetime

In [9]:
model = ECAPA_TDNN(input_size = 2, channels=[512,512,512,512,1024], lin_neurons=192, out_neurons=5, device = "cuda", metrics_type="cosine")
#model = ECAPA_TDNN(input_size = 8, lin_neurons=6, device = "cpu")
model.train()
#input_feats = torch.rand([10,2,512])
#print(input_feats)
#output = model(input_feats)
#output.shape

ECAPA_TDNN(
  (blocks): ModuleList(
    (0): TDNNBlock(
      (conv): Conv1d(
        (conv): Conv1d(2, 512, kernel_size=(5,), stride=(1,))
      )
      (activation): ReLU()
      (norm): BatchNorm1d(
        (norm): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (1): SERes2NetBlock(
      (tdnn1): TDNNBlock(
        (conv): Conv1d(
          (conv): Conv1d(512, 512, kernel_size=(1,), stride=(1,))
        )
        (activation): ReLU()
        (norm): BatchNorm1d(
          (norm): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
      )
      (res2net_block): Res2NetBlock(
        (blocks): ModuleList(
          (0-6): 7 x TDNNBlock(
            (conv): Conv1d(
              (conv): Conv1d(64, 64, kernel_size=(3,), stride=(1,), dilation=(2,))
            )
            (activation): ReLU()
            (norm): BatchNorm1d(
              (norm): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=T

In [3]:
with h5py.File('../RadChar-Tiny.h5', 'r') as f:
    # Print the dataset names
    print(list(f.keys())) # OUT: ['iq', 'labels']
    
    # Get a reference to the dataset
    h5_iqs = f['iq'] 
    h5_labels = f['labels'] 
    
    # Print the shape of the dataset
    # print(h5_iqs.shape) # OUT: (50000, 512)
    # print(h5_labels.shape) # OUT: (50000,)
    
    # Print the contents of the dataset
    loaded_h5_iqs = h5_iqs[...] # IQ data
    loaded_h5_labels = h5_labels[...] # Label data

# Temporary fix to deal with IQ data
temp_iq_data = torch.tensor(loaded_h5_iqs)
t_iq_real = temp_iq_data.real.float()
t_iq_imag = temp_iq_data.imag.float()
t_iq_data = torch.stack([t_iq_real, t_iq_imag], dim = 1)
print(t_iq_data.shape)

index = loaded_h5_labels['index']
signal_type = loaded_h5_labels['signal_type']
t_index = torch.tensor(index, dtype=torch.int64)
t_signal_type = torch.tensor(signal_type, dtype=torch.int64)
combined = torch.stack([t_index, t_signal_type], dim=1)
dataset = TensorDataset(t_iq_data, combined)

['iq', 'labels']
torch.Size([50000, 2, 512])


In [6]:
# total size
total_size = 50000
train_size = 45000
val_size = total_size - train_size
# batch size
batch_size = 32

# split
train_dataset, val_dataset = random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)  # for reproducibility
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print("Done")

Done


# Run the below only for training, you can vary num_epochs accordingly

In [ ]:
model.train()
training = BaseTrainer(model)
training._train(train_loader, num_epochs=2)

# Get current date and time formatted as YYYY-MM-DD_HH-MM-SS
now = datetime.now()
timestamp = now.strftime("%Y%m%d_%H:%M:%S")
filename = f"model_weights_{timestamp}.pt"
training.save_model(filename, model)

# Run the below only for testing, you can change the loaded .pt file accordingly

In [7]:
checkpoint = torch.load("./model_weights_.pt", weights_only=True, map_location='cpu')
model_test = ECAPA_TDNN(input_size = 2, channels=[512,512,512,512,1024], lin_neurons=192, out_neurons=5, device = "cuda", metrics_type="cosine")
model_test.load_state_dict(checkpoint)
model_test.to('cuda')
model_test.eval()
testing = BaseTrainer(model_test)
testing._test(val_loader)

Batch No. 2/157:   0%|                                                                          | 0/157 [00:00<?, ?it/s]

Batch No. 4/157:   0%|                                                                          | 0/157 [00:00<?, ?it/s]

Batch No. 6/157:   0%|                                                                          | 0/157 [00:00<?, ?it/s]

Batch No. 8/157:   0%|                                                                          | 0/157 [00:00<?, ?it/s]

Batch No. 10/157:   0%|                                                                         | 0/157 [00:00<?, ?it/s]

Batch No. 12/157:   0%|                                                                         | 0/157 [00:00<?, ?it/s]

Batch No. 14/157:   0%|                                                                         | 0/157 [00:00<?, ?it/s]

Batch No. 16/157:   0%|                                                                         | 0/157 [00:00<?, ?it/s]

Batch No. 18/157:   0%| 


Test set: Average loss: 0.9321, Accuracy: 4245/5000 (85%)

Time : 22.786937364959158


array([[956.,  15.,  28.,  34.,  30.,   0.],
       [  2., 831.,  24.,  25.,  24.,   0.],
       [  3.,  57., 782.,  48.,  49.,   0.],
       [ 46.,  89., 129., 855., 112.,   0.],
       [  0.,  15.,  16.,   9., 821.,   0.],
       [  0.,   0.,   0.,   0.,   0.,   0.]])